In [1]:
import sys
import os
sys.path.append(os.path.abspath('../../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.ensembles import *

from sklearn import set_config
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import make_scorer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold
from sksurv.metrics import integrated_brier_score
from sksurv.metrics import concordance_index_ipcw
from sksurv.nonparametric import kaplan_meier_estimator
from sklearn.inspection import permutation_importance

from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

import time
set_config(display="text")  # displays text representation of estimators

In [2]:
dataset = pd.read_csv("../data/data_final.csv")

diagnosis_variables = []

with open("../data/diagnosis_final.txt", "r") as file:
    for line in file:
        diagnosis_variables.append(line.strip())

treatment_variables = []
with open("../data/treatment_final.txt", "r") as file:
    for line in file:
        treatment_variables.append(line.strip())

survival_dataset = dataset.drop('Unnamed: 0', axis=1)

feature_columns = treatment_variables + diagnosis_variables

In [3]:
# Find duplicated patients
mask = (survival_dataset.duplicated(subset='PERSON_ID', keep=False)) #& (survival_dataset['Regional LN recurrence'] == 1)

# For each duplicated PERSON_ID with recurrence, keep only the row with the lowest survival days
to_remove = (
    survival_dataset[mask]
    .sort_values(['PERSON_ID', 'Survival days'], ascending=[True, True])
    .groupby('PERSON_ID')
    .head(-1)
    .index
)

survival_dataset = survival_dataset.drop(to_remove)

In [4]:
# Prepare data
data_x = survival_dataset[feature_columns]
data_y = Surv.from_arrays(
    event=survival_dataset['Regional LN recurrence'],
    time=survival_dataset['Survival days']
)

In [5]:
sgkf = StratifiedKFold(n_splits=5, shuffle=True, random_state=12)
cv = sgkf.split(X=data_x, y=survival_dataset['Regional LN recurrence'])


train_index, test_index = next(cv)
x_train = data_x.iloc[train_index]
y_train = data_y[train_index]

x_test = data_x.iloc[test_index]
y_test = data_y[test_index]

subsampling = survival_dataset.iloc[train_index].copy()

In [43]:
for train_idx, val_idx in cv:
    print("Train events:", survival_dataset.iloc[train_idx]["Regional LN recurrence"].sum())
    print("Val events:", survival_dataset.iloc[val_idx]["Regional LN recurrence"].sum())

Train events: 27.0
Val events: 6.0
Train events: 26.0
Val events: 7.0
Train events: 26.0
Val events: 7.0
Train events: 26.0
Val events: 7.0


In [44]:
'''data_x = survival_dataset[feature_columns]
data_y = Surv.from_arrays(
    event=survival_dataset['Regional LN recurrence'],
    time=survival_dataset['Survival days']
)
groups = survival_dataset['PERSON_ID']
event_indicator = survival_dataset['Regional LN recurrence']

time = data_y["time"]
valid_mask = time < 0

data_x = data_x[valid_mask]
data_y = data_y[valid_mask]
groups = groups[valid_mask]
event_indicator = event_indicator[valid_mask]'''

'data_x = survival_dataset[feature_columns]\ndata_y = Surv.from_arrays(\n    event=survival_dataset[\'Regional LN recurrence\'],\n    time=survival_dataset[\'Survival days\']\n)\ngroups = survival_dataset[\'PERSON_ID\']\nevent_indicator = survival_dataset[\'Regional LN recurrence\']\n\ntime = data_y["time"]\nvalid_mask = time < 0\n\ndata_x = data_x[valid_mask]\ndata_y = data_y[valid_mask]\ngroups = groups[valid_mask]\nevent_indicator = event_indicator[valid_mask]'

In [ ]:
'''pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", FastSurvivalSVM(random_state=12))
])'''

param_grid = {
    'alpha': [0.01, 0.1, 1.0, 10.0, 100.0],
    'rank_ratio': [0.0, 0.5, 1.0],
    'tol': [1e-3, 1e-4],
    'max_iter': [5000, 10000],
}

# Best parameters: {'svm__alpha': 0.01, 'svm__max_iter': 5000, 'svm__rank_ratio': 1.0, 'svm__tol': 0.0001}


In [46]:
sgkf_1 = StratifiedKFold(n_splits=4, shuffle=True, random_state=12)
cv = sgkf_1.split(X=x_train, y=y_train['event'])

In [ ]:
results_df, best_params, summary = generic_grid_search(
    FastSurvivalSVM,
    x_train,
    y_train,
    cv,
    param_grid,
    metrics=['c-index', 'c-index-ipcw'], #it can't measure int-brier-score since it doesnt prodce surv curves
    model_name='SVM',
    scale_data=True,
    scaler_type='standard',
    verbose=True,
    random_state=12,
    n_jobs=-1
    )


SVM GRID SEARCH (Manual Implementation)
Model: FastSurvivalSVM
Scaling: Yes (standard) - INSIDE each fold
Metrics: ['c-index', 'c-index-ipcw']
Parameter Grid:
  - alpha: [0.01, 0.1, 1.0, 10.0, 100.0]
  - rank_ratio: [0.0, 0.5, 1.0]
  - tol: [0.001, 0.0001]
  - optimizer: ['avltree']
  - max_iter: [5000, 10000]
Total combinations: 60

[1/60] Testing params: {'alpha': 0.01, 'rank_ratio': 0.0, 'tol': 0.001, 'optimizer': 'avltree', 'max_iter': 5000}


  c-index: 0.5889 ± 0.1437
  c-index-ipcw: 0.5827 ± 0.1625
[2/60] Testing params: {'alpha': 0.01, 'rank_ratio': 0.0, 'tol': 0.001, 'optimizer': 'avltree', 'max_iter': 10000}
  Elapsed: 0.0min, Est. remaining: 1.4min
  c-index: 0.5889 ± 0.1437
  c-index-ipcw: 0.5827 ± 0.1625
[3/60] Testing params: {'alpha': 0.01, 'rank_ratio': 0.0, 'tol': 0.0001, 'optimizer': 'avltree', 'max_iter': 5000}
  Elapsed: 0.0min, Est. remaining: 1.2min
  c-index: 0.5893 ± 0.1440
  c-index-ipcw: 0.5839 ± 0.1631
[4/60] Testing params: {'alpha': 0.01, 'rank_ratio': 0.0, 'tol': 0.0001, 'optimizer': 'avltree', 'max_iter': 10000}
  Elapsed: 0.1min, Est. remaining: 1.5min
  c-index: 0.5893 ± 0.1440
  c-index-ipcw: 0.5839 ± 0.1631
[5/60] Testing params: {'alpha': 0.01, 'rank_ratio': 0.5, 'tol': 0.001, 'optimizer': 'avltree', 'max_iter': 5000}
  Elapsed: 0.1min, Est. remaining: 1.6min
  c-index: 0.2515 ± 0.0989
  c-index-ipcw: 0.3425 ± 0.1760
[6/60] Testing params: {'alpha': 0.01, 'rank_ratio': 0.5, 'tol': 0.001, 'opti

In [55]:
# View the results DataFrame
results_df.sort_values(by='c-index_mean', ascending=False)

,alpha,rank_ratio,tol,optimizer,max_iter,c-index_mean,c-index_std,c-index-ipcw_mean,c-index-ipcw_std
8,0.01,1.0,0.0010,avltree,5000,0.717424,0.126367,0.675244,0.164976
9,0.01,1.0,0.0010,avltree,10000,0.717424,0.126367,0.675244,0.164976
11,0.01,1.0,0.0001,avltree,10000,0.717109,0.126746,0.675220,0.165003
10,0.01,1.0,0.0001,avltree,5000,0.717109,0.126746,0.675220,0.165003
59,100.00,1.0,0.0001,avltree,10000,0.704892,0.139107,0.694739,0.140029
58,100.00,1.0,0.0001,avltree,5000,0.704892,0.139107,0.694739,0.140029
57,100.00,1.0,0.0010,avltree,10000,0.704643,0.143634,0.692907,0.146379
56,100.00,1.0,0.0010,avltree,5000,0.704643,0.143634,0.692907,0.146379
44,10.00,1.0,0.0010,avltree,5000,0.704213,0.143155,0.695243,0.142795
45,10.00,1.0,0.0010,avltree,10000,0.704213,0.143155,0.695243,0.142795


In [56]:
results_df.sort_values(by='c-index_mean', ascending=False).to_csv("../results/svm_grid_search_results.csv", index=False)

# Train and Test Final Model with Best Hyperparameters

In [7]:
# Load best hyperparameters from grid search results
best_results = pd.read_csv("../results/normal/svm_grid_search_results.csv")
# Select row with highest c-index-ipcw_mean
best_params = best_results.sort_values('c-index-ipcw_mean', ascending=False).iloc[0]

print("Best hyperparameters:")
print(f"  alpha: {best_params['alpha']}")
print(f"  rank_ratio: {best_params['rank_ratio']}")
print(f"  tol: {best_params['tol']}")
print(f"  max_iter: {int(best_params['max_iter'])}")
print(f"  c-index-ipcw_mean (CV): {best_params['c-index-ipcw_mean']:.4f}")
print()

Best hyperparameters:
  alpha: 10.0
  rank_ratio: 1.0
  tol: 0.001
  max_iter: 5000
  c-index-ipcw_mean (CV): 0.6952



In [8]:
# Train final model with best hyperparameters (NO SUBSAMPLING)
from sksurv.svm import FastSurvivalSVM
from sklearn.preprocessing import StandardScaler

# Scale the full training set (SVM requires scaling)
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)

final_model = FastSurvivalSVM(
    alpha=best_params['alpha'],
    rank_ratio=best_params['rank_ratio'],
    tol=best_params['tol'],
    max_iter=int(best_params['max_iter']),
    random_state=12
)

final_model.fit(x_train_scaled, y_train)
print("Model trained successfully on full training set (no subsampling)!")

Model trained successfully on full training set (no subsampling)!


In [9]:
# Test the model on test set
x_test_scaled = scaler.transform(x_test)

# Calculate c-index
c_index = final_model.score(x_test_scaled, y_test)

# Calculate c-index IPCW
tau = y_train["time"].max()
c_index_ipcw = concordance_index_ipcw(y_train, y_test, final_model.predict(x_test_scaled), tau=tau)[0]

print("\nTest Set Performance (No Subsampling):")
print(f"  C-index: {c_index:.4f}")
print(f"  C-index IPCW: {c_index_ipcw:.4f}")
print("  Note: FastSurvivalSVM does not produce survival functions, so Brier score cannot be calculated.")


Test Set Performance (No Subsampling):
  C-index: 0.7323
  C-index IPCW: 0.6870
  Note: FastSurvivalSVM does not produce survival functions, so Brier score cannot be calculated.
